# Module 4.1 — First-Class Functions & Closures

### Python Mastery · Track 4: Functional Programming & Metaprogramming

In Python, functions are ordinary values: you can store them, pass them, and return them like any other object. This unlocks a flexible style of programming and is the foundation of decorators and much of the functional toolkit. This module covers first-class functions, higher-order functions, and closures.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it (`Shift + Enter`).
- Pass functions around and inspect what they capture.
- Attempt the exercises before consulting the solutions at the end.

### Learning objectives

After completing this module you will be able to:

1. Treat functions as values: assign, store, and pass them.
2. Write higher-order functions that take or return functions.
3. Explain what a closure is and how it captures variables.
4. Use `nonlocal` to maintain state inside a closure.
5. Recognise and avoid the late-binding capture pitfall.

**Prerequisites:** Tracks 1 to 3 (especially Module 1.4).

---

## Part 1 · Functions Are Objects

A function defined with `def` is an object like any other. It has a name and attributes, and a variable can refer to it. Assigning a function to another name does not call it; it simply creates a second reference to the same function object. You call a function only with parentheses.

In [ ]:
def greet(name):
    """Return a greeting."""
    return f"Hello, {name}"

# Assign the function object to another name (no parentheses, so no call yet).
f = greet
print("same object:", f is greet)
print("calling via the new name:", f("Asha"))

# Functions have attributes, like any object.
print("name:", greet.__name__)
print("docstring:", greet.__doc__)

Because functions are values, they can be stored in data structures such as lists and dictionaries. A dictionary of functions is a clean alternative to a long chain of `if`/`elif`.

In [ ]:
def add(a, b): return a + b
def sub(a, b): return a - b
def mul(a, b): return a * b

# A dictionary mapping a name to a function: a small dispatch table.
operations = {"+": add, "-": sub, "*": mul}

for symbol in ["+", "-", "*"]:
    func = operations[symbol]            # look up the function
    print(f"6 {symbol} 2 = {func(6, 2)}")  # then call it

## Part 2 · Higher-Order Functions

A **higher-order function** is one that takes a function as an argument, returns a function, or both. Taking a function as an argument lets you parameterise behaviour, the caller decides what to do with each item, while your function decides when and how often.

In [ ]:
def apply_twice(func, value):
    """Apply func to value two times."""
    return func(func(value))

print("add 3 twice to 10:", apply_twice(lambda x: x + 3, 10))   # 16
print("square twice:", apply_twice(lambda x: x * x, 2))          # 16

def transform_all(func, items):
    """Apply func to every item, returning a new list."""
    return [func(item) for item in items]

print("uppercased:", transform_all(str.upper, ["a", "b", "c"]))
print("doubled:", transform_all(lambda n: n * 2, [1, 2, 3]))

## Part 3 · Closures

A **closure** is a function defined inside another function that remembers the variables from the enclosing scope, even after the outer function has finished. The inner function "closes over" those variables, keeping them alive. This is how a function can carry private, persistent data without a class.

In [ ]:
def make_multiplier(factor):
    """Return a function that multiplies its argument by 'factor'."""
    def multiply(value):
        return value * factor            # 'factor' is captured from the enclosing scope
    return multiply                       # return the inner function itself

double = make_multiplier(2)
triple = make_multiplier(3)

print("double(5):", double(5))           # 10, remembers factor=2
print("triple(5):", triple(5))           # 15, remembers factor=3

# The captured value persists after make_multiplier has returned.
print("double still works:", double(10))

You can confirm that a closure has captured variables by inspecting `__closure__`. Each captured variable is stored in a cell, and the function records which names are free (defined in an enclosing scope).

In [ ]:
def make_greeter(greeting):
    def greet(name):
        return f"{greeting}, {name}"
    return greet

hello = make_greeter("Hello")
print(hello("Asha"))

# Inspect what was captured.
print("free variables:", hello.__code__.co_freevars)
print("captured value:", hello.__closure__[0].cell_contents)

## Part 4 · Maintaining State with `nonlocal`

A closure can read enclosing variables freely, but to **reassign** one it must declare it `nonlocal`. This lets a closure keep mutable state between calls, such as a running total or a counter, which is a lightweight alternative to a class for simple cases.

In [ ]:
def make_counter():
    count = 0
    def increment():
        nonlocal count          # without this, 'count += 1' would create a new local
        count += 1
        return count
    return increment

counter = make_counter()
print(counter())   # 1
print(counter())   # 2
print(counter())   # 3

# Each counter has independent state.
other = make_counter()
print("independent counter:", other())   # 1

## Part 5 · The Late-Binding Capture Pitfall

A subtle trap: a closure captures the **variable**, not the value at the time of definition. If you create functions in a loop that all close over the loop variable, they will all see its **final** value. The fix is to bind the current value explicitly, commonly via a default argument, which is evaluated when the function is defined.

In [ ]:
# The trap: all functions print 2, because they share the same 'i', read at call time.
buggy = []
for i in range(3):
    buggy.append(lambda: i)
print("buggy (all the same):", [f() for f in buggy])   # [2, 2, 2]

# The fix: capture the current value with a default argument.
fixed = []
for i in range(3):
    fixed.append(lambda i=i: i)        # 'i=i' binds the value now, at definition
print("fixed:", [f() for f in fixed])  # [0, 1, 2]

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Storing functions in a dictionary (Easy)

In [ ]:
def area_square(s): return s * s
def area_circle(r): return 3.14159 * r * r

shapes = {"square": area_square, "circle": area_circle}
print("square:", shapes["square"](4))
print("circle:", round(shapes["circle"](2), 2))
# Experiment: add a "triangle" entry with its own function.

### Example 2 — A higher-order function (Easy)

In [ ]:
def repeat(func, times, value):
    """Apply func to value 'times' times in succession."""
    for _ in range(times):
        value = func(value)
    return value

print("add 1 five times to 0:", repeat(lambda x: x + 1, 5, 0))   # 5
print("double three times:", repeat(lambda x: x * 2, 3, 1))       # 8

### Example 3 — A closure that configures behaviour (Medium)

In [ ]:
def make_power(exponent):
    """Return a function that raises its argument to 'exponent'."""
    def power(base):
        return base ** exponent
    return power

square = make_power(2)
cube = make_power(3)
print("square(5):", square(5))
print("cube(2):", cube(2))
print("they are independent:", square(3), cube(3))

### Example 4 — A stateful accumulator with nonlocal (Medium)

In [ ]:
def make_accumulator():
    """Return a function that adds each value to a running total and returns it."""
    total = 0
    def add(amount):
        nonlocal total
        total += amount
        return total
    return add

acc = make_accumulator()
print(acc(10))    # 10
print(acc(5))     # 15
print(acc(100))   # 115

### Example 5 — Function composition (Difficult)

In [ ]:
def compose(*functions):
    """Return a function that applies the given functions right to left."""
    def composed(value):
        for func in reversed(functions):     # rightmost function runs first
            value = func(value)
        return value
    return composed

add_one = lambda x: x + 1
double = lambda x: x * 2
to_str = lambda x: f"result={x}"

# Equivalent to to_str(double(add_one(value)))
pipeline = compose(to_str, double, add_one)
print(pipeline(5))      # add_one -> 6, double -> 12, to_str -> "result=12"

### Example 6 — A memoising closure (Difficult)

In [ ]:
def make_memoised(func):
    """Wrap func so repeated calls with the same argument are cached."""
    cache = {}                              # private state held in the closure
    def wrapper(n):
        if n not in cache:
            cache[n] = func(n)              # compute and store on first sight
        return cache[n]
    return wrapper

calls = []
def slow_square(n):
    calls.append(n)                          # record each real computation
    return n * n

fast = make_memoised(slow_square)
print(fast(4), fast(4), fast(5))             # 16 16 25
print("actually computed for:", calls)       # [4, 5] -- 4 computed only once

---

## Exercises

**Exercise 1 (Easy).** Create a dictionary mapping the strings `"upper"`, `"lower"`, and `"title"` to the corresponding string methods, then use it to transform the word `"hello world"` three ways.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a higher-order function `apply_to_pair(func, a, b)` that returns `func(a, b)`. Call it with addition and with multiplication using lambdas.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a function `make_adder(n)` that returns a function adding `n` to its argument. Create an `add_ten` and an `add_hundred` and demonstrate both.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a closure `make_running_max()` that returns a function; each call passes a number and the function returns the largest value seen so far. Demonstrate the running maximum over a few calls.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** The loop below intends to build three functions that return 0, 10, and 20 respectively, but all return 20 due to late binding. Fix it so they return the intended values.

In [ ]:
funcs = []
for n in [0, 10, 20]:
    funcs.append(lambda: n)
# Fix the loop above, then this should print [0, 10, 20]:
print([f() for f in funcs])


---

## Solutions

**Solution 1**

In [ ]:
methods = {"upper": str.upper, "lower": str.lower, "title": str.title}
word = "hello world"
for name, method in methods.items():
    print(name, "->", method(word))

**Solution 2**

In [ ]:
def apply_to_pair(func, a, b):
    return func(a, b)

print(apply_to_pair(lambda x, y: x + y, 3, 4))
print(apply_to_pair(lambda x, y: x * y, 3, 4))

**Solution 3**

In [ ]:
def make_adder(n):
    def add(x):
        return x + n
    return add

add_ten = make_adder(10)
add_hundred = make_adder(100)
print(add_ten(5), add_hundred(5))

**Solution 4**

In [ ]:
def make_running_max():
    highest = None
    def update(value):
        nonlocal highest
        if highest is None or value > highest:
            highest = value
        return highest
    return update

rm = make_running_max()
for v in [3, 7, 2, 9, 5]:
    print(v, "->", rm(v))

**Solution 5**

In [ ]:
funcs = []
for n in [0, 10, 20]:
    funcs.append(lambda n=n: n)    # bind the current value via a default argument
print([f() for f in funcs])

---

## Summary and Key Points

- Functions are first-class objects: they can be assigned to names, stored in containers (a dict of functions is a clean dispatch table), and passed around. Parentheses call a function; a bare name refers to it.
- Higher-order functions take or return functions, letting you parameterise behaviour.
- A closure is an inner function that captures variables from its enclosing scope and keeps them alive after the outer function returns, providing private persistent state without a class.
- `nonlocal` allows a closure to reassign an enclosing variable, enabling stateful closures such as counters and accumulators.
- Closures capture variables, not values; functions created in a loop share the loop variable. Bind the current value explicitly (commonly with a default argument) to avoid the late-binding pitfall.

### Next module: 4.2 — Decorators Deep Dive

The next module builds directly on closures to create decorators: functions that wrap other functions to add behaviour, including parameterised and stacked decorators.